In [33]:
from pathlib import Path
import polars as pl
from dataclasses import dataclass


CATEGORY = "Amazon_Fashion"
DATA_DIR = Path.cwd() / "data"

review_path = DATA_DIR / f"review_{CATEGORY}.jsonl"
meta_path   = DATA_DIR / f"meta_{CATEGORY}.jsonl"
train_path  = DATA_DIR / f"{CATEGORY}.train.csv"


## Подготовка информации о товарах

In [18]:
@dataclass(frozen=True)
class MetaPrepConfig:
    min_title_len: int = 20
    min_desc_len: int = 100

META_SCHEMA = [
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "features",
    "description",
    "price",
    "store",
]  # для построения эмббедингов не требуются images, videos, bought_together, details, а categories пустые

def _read_meta_lazy(meta_jsonl: Path) -> pl.LazyFrame:
    return pl.scan_ndjson(meta_jsonl, ignore_errors=True).select([pl.col(c) for c in META_SCHEMA])

def _normalize_meta_types(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.select(
        pl.col("parent_asin").cast(pl.Utf8),
        pl.col("main_category").cast(pl.Utf8),
        pl.col("title").cast(pl.Utf8),
        pl.col("average_rating").cast(pl.Float32),
        pl.col("rating_number").cast(pl.Int32),
        pl.col("features").cast(pl.List(pl.Utf8)),
        pl.col("description").cast(pl.List(pl.Utf8)),
        pl.col("price").cast(pl.Float32),
        pl.col("store").cast(pl.Utf8),
    )

def _add_text_fields(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.col("description").list.join(" ").fill_null("").alias("description_text"),
        pl.col("features").list.join(" ").fill_null("").alias("features_text"),
    )

def _filter_valid_items(lf: pl.LazyFrame, cfg) -> pl.LazyFrame:
    return lf.filter(
        pl.col("parent_asin").is_not_null() &
        pl.col("title").is_not_null() &
        (pl.col("title").str.len_chars() > cfg.min_title_len) &
        (pl.col("description_text").str.len_chars() > cfg.min_desc_len)
    )

def _fill_nulls(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.col("main_category").fill_null(""),
        pl.col("title").fill_null(""),
        pl.col("features_text").fill_null(""),
        pl.col("description_text").fill_null(""),
        pl.col("store").fill_null(""),
        pl.col("average_rating").fill_null(pl.lit(None, dtype=pl.Float32)),
        pl.col("rating_number").fill_null(0),
        pl.col("price").fill_null(pl.lit(None, dtype=pl.Float32)),
    )

def _add_item_context(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.concat_str(
            [
                pl.lit("Product: "), pl.col("title"),
                pl.lit("\n\nDescription: "), pl.col("description_text"),
                pl.lit("\n\nFeatures: "), pl.col("features_text"),
                pl.lit("\n\nCategory: "), pl.col("main_category"),
                pl.lit("\n\nStore: "), pl.col("store"),
                pl.lit("\n\nAverage rating: "), pl.col("average_rating").cast(pl.Utf8).fill_null(""),
                pl.lit(", Rating count: "), pl.col("rating_number").cast(pl.Utf8),
                pl.lit("\n\nPrice: "), pl.col("price").cast(pl.Utf8).fill_null(""),
            ]
        ).alias("item_context")
    )

In [19]:
def prepare_meta_dataset(
    meta_jsonl: Path,
    cfg: MetaPrepConfig = MetaPrepConfig(),
) -> tuple[pl.DataFrame, pl.Series]:
    """
    Returns:
      item_df: cleaned dataframe with item_context
      valid_items: pl.Series of parent_asin (unique)
    """

    lf = _read_meta_lazy(meta_jsonl)
    lf = _normalize_meta_types(lf)
    lf = _add_text_fields(lf)
    lf = _filter_valid_items(lf, cfg)

    # оставляем только нужное
    lf = lf.select(
        "parent_asin",
        "main_category",
        "title",
        "average_rating",
        "rating_number",
        "features_text",
        "description_text",
        "price",
        "store",
    )

    lf = _fill_nulls(lf)
    lf = _add_item_context(lf)

    item_df = lf.collect(streaming=True)
    valid_items = item_df.get_column("parent_asin").unique()

    return item_df, valid_items

In [72]:
item_df, valid_items = prepare_meta_dataset(meta_path)

/var/folders/l_/4np1fb3j1lx613c5qvnrj8xw0000gn/T/ipykernel_46974/3393983472.py:32: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  item_df = lf.collect(streaming=True)


## Подготовка информации об отзывах

In [ ]:
def load_reviews_table(reviews_jsonl: Path) -> pl.DataFrame:
    return (
        pl.scan_ndjson(reviews_jsonl)
        .select(
            pl.col("user_id"),
            pl.col("parent_asin"),
            pl.col("rating").cast(pl.Float32),
            pl.col("title"),
            pl.col("text"),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("verified_purchase"),
            pl.col("helpful_vote").cast(pl.Int32),
        )
        .drop_nulls(["user_id", "parent_asin", "rating"])
        .collect(streaming=True)
    )

In [29]:
reviews = load_reviews_table(review_path)

/var/folders/l_/4np1fb3j1lx613c5qvnrj8xw0000gn/T/ipykernel_93066/3325490328.py:18: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


In [30]:
reviews

user_id,parent_asin,rating,title,text,timestamp,verified_purchase,helpful_vote
str,str,f32,str,str,i64,bool,i32
"""AGBFYI2DDIKXC5Y4FARTYDTQBMFQ""","""B00LOPVX74""",5.0,"""Pretty locket""","""I think this locket is really …",1578528394489,true,3
"""AFQLNQNQYFWQZPJQZS6V3NZU4QBQ""","""B07B4JXK8D""",5.0,"""A""","""Great""",1608426246701,true,0
"""AHITBJSS7KYUBVZPX7M2WJCOIVKQ""","""B007ZSEQ4Q""",2.0,"""Two Stars""","""One of the stones fell out wit…",1432344828000,true,3
"""AFVNEEPDEIH5SPUN5BWC6NKL3WNQ""","""B07F2BTFS9""",1.0,"""Won’t buy again""","""Crappy socks. Money wasted. Bo…",1546289847095,true,2
"""AHSPLDNW5OOUK2PLH7GXLACFBZNQ""","""B00XESJTDE""",5.0,"""I LOVE these glasses""","""I LOVE these glasses! They fi…",1439476166000,true,0
…,…,…,…,…,…,…,…
"""AFXSFD3FTZ2CLN3TYV4B63CQM5BQ""","""B00YGFMQC0""",5.0,"""... allowed them to be used to…","""The tie tacks were the size th…",1466799158000,true,0
"""AEH7WP5HGM6FGLSSC6GSTYUXBHGQ""","""B00YGFMQC0""",1.0,"""Didn’t come with all ten""","""Says ten tie clips but o only …",1525799105585,true,0
"""AEL2TSSBVLIPWQ7YVMK364DUYURQ""","""B00YGFMQC0""",3.0,"""Not checked for quality""","""When I received them 2-3 of th…",1482013711000,true,0


## Подготовка последовательностей

In [64]:
# Load the gzipped CSV file
df = pl.read_csv(train_path)

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns}")


df = df.filter(
    pl.col("history").is_not_null() &
    (pl.col("history").str.len_chars() > 0)
)

df.head()

Dataset shape: (157490, 5)
Columns: ['user_id', 'parent_asin', 'rating', 'timestamp', 'history']


user_id,parent_asin,rating,timestamp,history
str,str,f64,i64,str
"""AHREXOGQPZDA6354MHH4ETSF3MCQ""","""B097DQPCP2""",5.0,1627391870221,"""B092J4ZT1V"""
"""AHREXOGQPZDA6354MHH4ETSF3MCQ""","""B089PWHFVW""",3.0,1628867386110,"""B092J4ZT1V B097DQPCP2"""
"""AFSKPY37N3C43SOI5IEXEK5JSIYA""","""B07H1BKYPT""",5.0,1549403115093,"""B014IQAIXA"""
"""AFSKPY37N3C43SOI5IEXEK5JSIYA""","""B07H92ZCQ9""",3.0,1549712804435,"""B014IQAIXA B07H1BKYPT"""
"""AFSKPY37N3C43SOI5IEXEK5JSIYA""","""B07NX5RHZ2""",5.0,1561338044117,"""B014IQAIXA B07H1BKYPT B07H92ZC…"


In [65]:
# Deduplicate by user_id, keeping the row with the longest history
# First, calculate the length of each history
df = df.with_columns(
    pl.when(pl.col("history").is_null())
    .then(0)
    .otherwise(pl.col("history").str.count_matches(r"\S+"))
    .alias("history_length")
)

print(f"Original dataset num rows: {df.shape[0]:,}")

# Sort by user_id and history_length (descending), then keep first row per user
df = df.sort(["user_id", "history_length"], descending=[False, True]).group_by("user_id").first().drop("history_length")

print(f"Deduplicated dataset num rows: {df.shape[0]:,}")
print(f"Number of unique users: {df.n_unique('user_id'):,}")

Original dataset num rows: 80,735
Deduplicated dataset num rows: 29,991
Number of unique users: 29,991


In [66]:
# Create sequences column by appending parent_asin to history as a list
df = df.with_columns(pl.col("history").str.split(" ").list.concat([pl.col("parent_asin")]).alias("sequence"))

df.head()

user_id,parent_asin,rating,timestamp,history,sequence
str,str,f64,i64,str,list[str]
"""AHLIVQGR5B7ANSWGWCCD6F3ZMLXQ""","""B00PJIQNRM""",4.0,1452185739000,"""B006RIQWOC B00X9DPJ7O""","[""B006RIQWOC"", ""B00X9DPJ7O"", ""B00PJIQNRM""]"
"""AHFJZARSHDAPPSVS72KITLMSRL5A""","""B016CUQI2A""",4.0,1482854017000,"""B01N3LK238""","[""B01N3LK238"", ""B016CUQI2A""]"
"""AHKEIHCCFYWIJPQEWOQBHX46RXPA""","""B06XKYMZPD""",5.0,1501070234407,"""B008D8L7L2 B00ZJX8V9A""","[""B008D8L7L2"", ""B00ZJX8V9A"", ""B06XKYMZPD""]"
"""AHPLRNHDHXEGQYTA3477IG3V4LRA""","""B01IY7ZW76""",5.0,1482368947000,"""B00HPT9XO8""","[""B00HPT9XO8"", ""B01IY7ZW76""]"
"""AE2B3VOH6CQDAK7KV5CV6T5LNJWQ""","""B010CA6P56""",5.0,1502989503344,"""B01BIEN0BM""","[""B01BIEN0BM"", ""B010CA6P56""]"


In [73]:
valid_items = set(valid_items.to_list())

In [77]:
# Function to filter sequence items based on valid metadata
def filter_sequence_items(sequence_list, valid_items_set):
    if sequence_list is None:
        return None

    # Filter the list to keep only items with valid metadata
    filtered_items = [item for item in sequence_list if item in valid_items_set]

    return filtered_items if filtered_items else None


# Filter sequences where metadata is missing
df = df.with_columns(
    pl.col("sequence")
    .map_elements(lambda x: filter_sequence_items(x, valid_items), return_dtype=pl.List(pl.String))
    .alias("sequence")
)

# Filter out rows where:
# 1. The target item (parent_asin) doesn't have valid metadata
# 2. The filtered sequences is empty or null
rows_before_filtering = df.shape[0]
logger.info(f"Rows before filtering: {rows_before_filtering:,}")

df = df.filter((pl.col("sequence").is_not_null()) & (pl.col("sequence").list.len() >= MIN_SEQUENCE_LENGTH))

# Log statistics
logger.info(f"Rows after filtering: {df.shape[0]:,}")
logger.info(
    f"Rows removed: {rows_before_filtering - df.shape[0]:,} ({(rows_before_filtering - df.shape[0]) / rows_before_filtering * 100:.1f}%)"
)


thread '<unnamed>' (3147633) panicked at crates/polars-python/src/map/series.rs:966:10:
could not get Series attribute '_s': PyErr { type: <class 'AttributeError'>, value: AttributeError("'NoneType' object has no attribute '_s'"), traceback: None }
--- PyO3 is resuming a panic after fetching a PanicException from Python. ---
Python stack trace below:


PanicException: could not get Series attribute '_s': PyErr { type: <class 'AttributeError'>, value: AttributeError("'NoneType' object has no attribute '_s'"), traceback: None }

PanicException: could not get Series attribute '_s': PyErr { type: <class 'AttributeError'>, value: AttributeError("'NoneType' object has no attribute '_s'"), traceback: None }

In [81]:
import polars as pl

def audit_df(df: pl.DataFrame, *, sample: int = 0) -> pl.DataFrame:
    rows = []
    for c in df.columns:
        s = df[c]
        nulls = s.null_count()

        empty_str = None
        empty_list = None

        if s.dtype == pl.Utf8:
            empty_str = df.select((pl.col(c).str.strip_chars() == "").sum()).item()
        elif isinstance(s.dtype, pl.List):
            empty_list = df.select((pl.col(c).list.len() == 0).sum()).item()

        rows.append((c, str(s.dtype), int(nulls), empty_str, empty_list))

    out = pl.DataFrame(rows, schema=["column", "dtype", "null_count", "empty_str_count", "empty_list_count"])

    if sample > 0:
        # покажем строки, где есть проблема
        issues = []
        for c in df.columns:
            if df[c].dtype == pl.Utf8:
                issues.append(pl.col(c).is_null() | (pl.col(c).str.strip_chars() == ""))
            elif isinstance(df[c].dtype, pl.List):
                issues.append(pl.col(c).is_null() | (pl.col(c).list.len() == 0))
            else:
                issues.append(pl.col(c).is_null())
        bad = df.filter(pl.any_horizontal(issues)).head(sample)
        print("Sample problematic rows:")
        print(bad)

    return out

In [83]:
audit_df(df)

/var/folders/l_/4np1fb3j1lx613c5qvnrj8xw0000gn/T/ipykernel_46974/2140771769.py:19: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  out = pl.DataFrame(rows, schema=["column", "dtype", "null_count", "empty_str_count", "empty_list_count"])


column,dtype,null_count,empty_str_count,empty_list_count
str,str,i64,i64,i64
"""user_id""","""String""",0,0,null
"""parent_asin""","""String""",0,0,null
"""rating""","""Float64""",0,null,null
"""timestamp""","""Int64""",0,null,null
"""history""","""String""",0,0,null
"""sequence""","""List(String)""",0,null,0
